**LightGBM and Statistical Benchmark models for Efat et al. (ATES + GRU-CNN-LSTM), 2022**

In [1]:
# Mount Google Drive
from google.colab import drive
#drive.mount('/content/drive',force_remount=True)
drive.mount('/content/drive')

Mounted at /content/drive


# Reference:

* Efat et al. (2024). Deep-learning model using hybrid adaptive trend
estimated series for modelling and forecasting sales.
* Annals of Operations Research*, 339, 297-328.
* https://doi.org/10.1007/s10479-022-04838-6

* **Dataset:** Corporación Favorita (Kaggle), monthly granularity

* **Series:** 162,503 item x store combinations with at least one recorded
transaction in train.csv during 2013-2016. The paper states 4,100 items x
54 stores but applies undisclosed data cleaning ("detecting and discarding
corrupted, incorrect, and incomplete samples", p.304), so the effective
series universe after cleaning is not recoverable.

* **Train:** January 2013 - December 2016

* **Test:** January - August 2017. h=12 is not feasible (Kaggle train.csv ends
15 August 2017, leaving only 8 test months).

* **Horizons:** h=1 (Jan), h=3 (Jan-Mar), h=6 (Jan-Jun) (cumulative evaluation)

* **LGBM model:** Recursive multi-step using a single h=1 model applied
iteratively through the 8 test months. After each monthly prediction the lag
state is updated exactly: lag_1 receives the prediction, lag_k receives the
previous lag_{k-1}, and the rolling statistics are recomputed from the updated
lag vector. Global model across all series, versus the paper's per store design
(54 models). seed=655321.

* **Statistical:** ADI-based classification on DAILY training data :
AutoETS (automatic model selection(seasonal and non-seasonal) based on least AICc per series) for smooth demand (ADI<1.32), SBA for
non-smooth (ADI>=1.32) and fall back is mean forecast if min_obs criteria is not satisfied

* **Prerequisites:** Kaggle Corporación Favorita files in 'DATA_DIR':
train.csv, stores.csv, items.csv, holidays_events.csv, oil.csv, transactions.csv

* Required packages: lightgbm, statsforecast, pandas, numpy, scikit-learn

**Import Libraries**

In [2]:
!pip install lightgbm statsforecast pandas numpy scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.0/501.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 10.0 MB/s eta 0:00:00


In [3]:
#load required packages
import os
import gc
import time
import logging
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, CrostonSBA

warnings.filterwarnings("ignore")

**Set paths (change as needed)**

In [20]:
##### paths #######################################
DATA_DIR    = "/content/drive/MyDrive/tll/raw_data"                                  # Kaggle corporacion favorita data CSVs
RESULTS_DIR = "/content/drive/MyDrive/tll_reproducibility/Dl using ATES and GRU_CNN_LSTM/monthly_benchmark"    # new outputs
CACHE_DIR   = os.path.join(RESULTS_DIR, "cache")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,   exist_ok=True)

MONTHLY_CSV = os.path.join(CACHE_DIR, "monthly_dataset.csv")

# logging - file handler plus console. force=True is required in Colab else it's a silent no-op
LOG_PATH = os.path.join(RESULTS_DIR, "run_log.txt")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.FileHandler(LOG_PATH, mode="a"), logging.StreamHandler()],
    force=True,
)
logger = logging.getLogger(__name__)
logger.info(f"Logging to {LOG_PATH}")

for f in ["train.csv", "stores.csv", "items.csv",
          "holidays_events.csv", "oil.csv", "transactions.csv"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"{f} not found in {DATA_DIR}"

11:19:48  Logging to /content/drive/MyDrive/tll_reproducibility/Dl using ATES and GRU_CNN_LSTM/monthly_benchmark/run_log.txt


**Configuration(Parameters)**

In [5]:
########## config #######
TRAIN_START = "2013-01-01"
TRAIN_END   = "2016-12-31"
TEST_START  = "2017-01-01"
TEST_END    = "2017-08-31"



ADI_THRESH = 1.32        # Syntetos & Boylan (2005)
SEED       = 655321
LAGS       = [1, 2, 3, 4, 5, 6]
ROLL_WINS  = [3, 6]
VAL_MONTHS = 3           # time-based validation: last 3 training months
HORIZONS   = {"1-month": 1, "3-month": 3, "6-month": 6}
MIN_OBS    = 2           # min monthly positive demand observations for statsforecast fitting as a crash guard

np.random.seed(SEED)

LGBM_PARAMS = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "learning_rate"    : 0.05,
    "num_leaves"       : 63,
    "max_depth"        : 6,
    "min_child_samples": 20,
    "subsample"        : 0.8,
    "bagging_freq"     : 1,
    "colsample_bytree" : 0.8,
    "verbose"          : -1,
    "seed"             : SEED,
    "n_jobs"           : -1,
}
NUM_ROUNDS     = 6000
EARLY_STOPPING = 100

logger.info(f"ADI  Threshold: {ADI_THRESH}")

09:26:27  ADI  Threshold: 1.32


**Import data**

In [6]:
##### load auxiliary data ##################

logger.info("Loading auxiliary data...")
t0 = time.time()

df_stores = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
df_items  = pd.read_csv(os.path.join(DATA_DIR, "items.csv"))

# holidays: national + regional (store state) + local (store city), monthly counts
df_hol = pd.read_csv(os.path.join(DATA_DIR, "holidays_events.csv"), parse_dates=["date"])
df_hol = df_hol[~df_hol["transferred"]].copy()
df_hol["month"] = df_hol["date"].dt.to_period("M").dt.to_timestamp()

nat = df_hol[df_hol.locale == "National"].groupby("month").size().reset_index(name="nat")
reg = df_hol[df_hol.locale == "Regional"]
loc = df_hol[df_hol.locale == "Local"]

store_hol = []
for _, s in df_stores.iterrows():
    h  = nat.copy()
    h["store_nbr"] = s["store_nbr"]
    r  = reg[reg.locale_name == s["state"]].groupby("month").size().reset_index(name="reg")
    lc = loc[loc.locale_name == s["city"]].groupby("month").size().reset_index(name="loc")
    h  = h.merge(r, on="month", how="left").merge(lc, on="month", how="left").fillna(0)
    h["holidays_count"] = h["nat"] + h["reg"] + h["loc"]
    store_hol.append(h[["store_nbr", "month", "holidays_count"]])
df_store_hol = pd.concat(store_hol, ignore_index=True)

# oil price: monthly mean, lagged one month so it is leakage-free at forecast time
df_oil = pd.read_csv(os.path.join(DATA_DIR, "oil.csv"), parse_dates=["date"])
df_oil["month"] = df_oil["date"].dt.to_period("M").dt.to_timestamp()
df_oil_m = df_oil.groupby("month")["dcoilwtico"].mean().reset_index(name="oil_price")
df_oil_m["oil_lag1"] = df_oil_m["oil_price"].shift(1)
df_oil_m = df_oil_m[["month", "oil_lag1"]]

# store transactions: monthly sum per store, lagged one month
df_trans = pd.read_csv(os.path.join(DATA_DIR, "transactions.csv"), parse_dates=["date"])
df_trans["month"] = df_trans["date"].dt.to_period("M").dt.to_timestamp()
df_trans_m = df_trans.groupby(["store_nbr", "month"])["transactions"].sum().reset_index()
df_trans_m = df_trans_m.sort_values(["store_nbr", "month"])
df_trans_m["transactions_lag1"] = df_trans_m.groupby("store_nbr")["transactions"].shift(1)
df_trans_m = df_trans_m[["store_nbr", "month", "transactions_lag1"]]

# label-encode store and item categoricals
le = LabelEncoder()
for col in ["city", "state", "type"]:
    df_stores[f"{col}_enc"] = le.fit_transform(df_stores[col].astype(str))
for col in ["family", "class"]:
    df_items[f"{col}_enc"] = le.fit_transform(df_items[col].astype(str))

logger.info(f"  Auxiliary data loaded in {time.time()-t0:.1f}s")

09:26:32  Loading auxiliary data...
09:26:35    Auxiliary data loaded in 2.4s


**Aggregate data to monthly level**

In [7]:
##### build monthly dataset ##################
# Aggregated straight from train.csv, so only item x store combinations with at
# least one recorded transaction appear

if os.path.exists(MONTHLY_CSV):
    logger.info("Cached monthly_dataset.csv found — loading")
    df = pd.read_csv(MONTHLY_CSV, parse_dates=["month"])
else:
    logger.info("Building monthly dataset from train.csv...")
    t0 = time.time()
    df_raw = pd.read_csv(
        os.path.join(DATA_DIR, "train.csv"), parse_dates=["date"],
        dtype={"store_nbr": int, "item_nbr": int,
               "unit_sales": float, "onpromotion": str}
    )
    df_raw["onpromotion"] = df_raw["onpromotion"].map(
        {"True": 1, "False": 0, True: 1, False: 0}).fillna(0).astype(int)
    df_raw["unit_sales"]  = df_raw["unit_sales"].clip(lower=0)
    df_raw["month"]       = df_raw["date"].dt.to_period("M").dt.to_timestamp()

    df = df_raw.groupby(["store_nbr", "item_nbr", "month"], as_index=False).agg(
        unit_sales_corrected=("unit_sales", "sum"),
        onpromotion=("onpromotion", "mean"),
    )
    df["unit_sales_corrected"] = df["unit_sales_corrected"].clip(lower=0)
    df.to_csv(MONTHLY_CSV, index=False)
    del df_raw
    gc.collect()
    logger.info(f"  Built in {(time.time()-t0)/60:.1f} min")

# merge auxiliary features
df = (df
    .merge(df_store_hol, on=["store_nbr", "month"], how="left")
    .merge(df_oil_m,     on="month",                how="left")
    .merge(df_trans_m,   on=["store_nbr", "month"], how="left")
    .merge(df_stores[["store_nbr", "city_enc", "state_enc", "type_enc", "cluster"]],
           on="store_nbr", how="left")
    .merge(df_items[["item_nbr", "family_enc", "class_enc", "perishable"]],
           on="item_nbr",  how="left")
)
df["holidays_count"]    = df["holidays_count"].fillna(0)
df["oil_lag1"]          = df["oil_lag1"].fillna(df["oil_lag1"].median())
df["transactions_lag1"] = df["transactions_lag1"].fillna(0)

n_series = df.groupby(["store_nbr", "item_nbr"]).ngroups
logger.info(f"  Series: {n_series:,}  Items: {df.item_nbr.nunique():,}  "
            f"Stores: {df.store_nbr.nunique()}  Months: {df.month.nunique()}")

09:26:38  Cached monthly_dataset.csv found — loading
09:26:41    Series: 174,685  Items: 4,036  Stores: 54  Months: 56


**Feature engineering**

In [8]:
##### feature engineering ##################
# 18 features: 6 monthly lags, rolling mean/std over 3 and 6 month windows,
# promotion and holiday counts (calendar-known ahead), oil and transactions
# lagged one month, store and item categoricals, and calendar

logger.info("Engineering features...")
t0 = time.time()

df = df.sort_values(["store_nbr", "item_nbr", "month"]).reset_index(drop=True)
df["y"]             = np.log1p(df["unit_sales_corrected"].clip(lower=0))
df["month_of_year"] = df["month"].dt.month
df["year"]          = df["month"].dt.year

for lag in LAGS:
    df[f"lag_{lag}"] = df.groupby(["store_nbr", "item_nbr"])["y"].shift(lag)

grp = df.groupby(["store_nbr", "item_nbr"]).ngroup()
for win in ROLL_WINS:
    # shift(1) before rolling so the current month is excluded from its own window
    shifted = df.groupby(["store_nbr", "item_nbr"])["y"].shift(1)
    df[f"roll_mean_{win}"] = shifted.groupby(grp).transform(
        lambda x: x.rolling(win, min_periods=1).mean())
    df[f"roll_std_{win}"]  = shifted.groupby(grp).transform(
        lambda x: x.rolling(win, min_periods=1).std().fillna(0))

FEATURES = (
    [f"lag_{l}"       for l in LAGS]      +
    [f"roll_mean_{w}" for w in ROLL_WINS]  +
    [f"roll_std_{w}"  for w in ROLL_WINS]  +
    ["onpromotion", "holidays_count", "oil_lag1", "transactions_lag1",
     "month_of_year", "year", "store_nbr", "item_nbr",
     "city_enc", "state_enc", "type_enc", "cluster",
     "family_enc", "class_enc", "perishable"]
)
LAG_ROLL = ([f"lag_{l}" for l in LAGS] +
            [f"roll_mean_{w}" for w in ROLL_WINS] +
            [f"roll_std_{w}"  for w in ROLL_WINS])

# classification source: monthly training rows captured BEFORE the feature
# dropna, so series with fewer than 7 months of history are still classified
df_class_src = df[df.month <= TRAIN_END][
    ["store_nbr", "item_nbr", "month", "unit_sales_corrected"]].copy()


n_before = len(df)
df = df.dropna(subset=FEATURES).reset_index(drop=True)
logger.info(f"  Dropped {n_before-len(df):,} rows with incomplete lag history")

# leakage check: roll_mean_3 must equal mean(lag_1, lag_2, lag_3)
sample = df[(df.store_nbr == df.store_nbr.iloc[0]) &
            (df.item_nbr  == df.item_nbr.iloc[0])].iloc[10:15]
for _, row in sample.iterrows():
    exp = np.mean([row["lag_1"], row["lag_2"], row["lag_3"]])
    if not np.isnan(exp):
        assert abs(row["roll_mean_3"] - exp) < 1e-5, "Rolling leakage detected"
logger.info("  Rolling leakage check passed")

df_train = df[df.month <= TRAIN_END].copy()
df_test  = df[(df.month >= TEST_START) & (df.month <= TEST_END)].copy()
logger.info(f"  Train: {len(df_train):,} rows  Test: {len(df_test):,} rows  "
            f"[{(time.time()-t0)/60:.1f} min]")

09:26:44  Engineering features...
09:27:42    Dropped 1,011,319 rows with incomplete lag history
09:27:42    Rolling leakage check passed
09:27:42    Train: 3,692,897 rows  Test: 1,109,088 rows  [1.0 min]


**Statistical Benchmarks: Sales Series classification**

In [9]:
###### Sales series (demand proxy) classification ##################
# ADI computed on the MONTHLY training series : the same granularity that is
# forecast, consistent with case studies 1 and 2

logger.info("Series classification (smooth/non-smooth) on monthly training data...")
t0 = time.time()

n_train_months = ((pd.Timestamp(TRAIN_END).to_period("M") -
                   pd.Timestamp(TRAIN_START).to_period("M")).n + 1)
logger.info(f"  ADI denominator: {n_train_months} training months")

sbc_rows = []
for (store, item), g in df_class_src.groupby(["store_nbr", "item_nbr"]):
    n_nz = int((g["unit_sales_corrected"].values > 0).sum())
    if n_nz == 0:
        cat, adi = "non-smooth", np.inf
    else:
        adi = n_train_months / n_nz
        cat = "smooth" if adi < ADI_THRESH else "non-smooth"
    sbc_rows.append({"store_nbr": store, "item_nbr": item,
                     "category": cat, "adi": round(float(adi), 4)})

df_sbc = pd.DataFrame(sbc_rows)

n_train_series = df_class_src.groupby(["store_nbr", "item_nbr"]).ngroups
assert len(df_sbc) == n_train_series, \
    f"classified {len(df_sbc):,} of {n_train_series:,} training series"
logger.info(f"  Classified {len(df_sbc):,} series with training history")
logger.info(f"  {n_series - n_train_series:,} series appear only in 2017 — "
            f"no statistical forecast possible, filled with 0 at evaluation")

del sbc_rows
gc.collect()

logger.info(f"  Classification done in {(time.time()-t0)/60:.1f} min")
logger.info(f"  Distribution:\n{df_sbc['category'].value_counts().to_string()}")

09:27:42  Series classification (smooth/non-smooth) on monthly training data...
09:27:42    ADI denominator: 48 training months
09:27:45    Classified 162,503 series with training history
09:27:45    12,182 series appear only in 2017 — no statistical forecast possible, filled with 0 at evaluation
09:27:45    Classification done in 0.0 min
09:27:45    Distribution:
category
non-smooth    99098
smooth        63405


**Statistical Benchmarks: model fitting**

In [ ]:
df_sbc.head()

,store_nbr,item_nbr,category,adi
0,1,96995,non-smooth,3.2000
1,1,99197,non-smooth,4.0000
2,1,103520,smooth,1.0213
3,1,103665,smooth,1.0000
4,1,105574,smooth,1.0000


In [14]:
######## statistical models #######################################
# Fitted on MONTHLY training data in ORIGINAL unit sales — statsforecast models
# expect counts, not log transformed values. AutoETS(season_length=12) performs an internal model search across error,
# trend, and seasonal components and selects the specification minimizing
# its own AICc. It does not force seasonality just because season_length>1,
# and returns a non-seasonal ETS specification where seasonality doesn't
# improve the fit. CrostonSBA is the Syntetos-Boylan bias-corrected Croston variant.

logger.info("Fitting statistical models...")
t0 = time.time()
stat_pred_path = os.path.join(RESULTS_DIR, "statistical_predictions_adi1.csv")
MAX_H = max(HORIZONS.values())
future_months = pd.date_range(
    start=pd.Timestamp(TRAIN_END) + pd.DateOffset(months=1), periods=MAX_H, freq="MS")

def is_seasonal(method_str):
    """ETS(E,T,S) notation"""
    return method_str.split(",")[-1].rstrip(")") != "N"

if os.path.exists(stat_pred_path):
    logger.info("  Cached statistical_predictions_adi.csv found — loading")
    df_stat_fc = pd.read_csv(stat_pred_path, parse_dates=["month"])
else:
    # Built from dataset before removing NAs
    df_stat = df_class_src.copy()
    df_stat["unique_id"] = (df_stat["store_nbr"].astype(str) + "_" +
                            df_stat["item_nbr"].astype(str))
    df_stat = df_stat.rename(columns={"month": "ds", "unit_sales_corrected": "y"})
    assert df_stat["y"].max() > 10, \
        "df_stat looks like log1p — check CELL 9 (log units vs original units)"
    df_stat = df_stat[["unique_id", "ds", "y"]]
    # Each series is reindexed from its own first observed month to December 2016.
    # Filling from January 2013 instead would fabricate pre-launch zeros for items
    # not yet stocked (2.6M rows, 33% of a full grid). Trailing zeros after discontinuation
    # are kept
    first_seen = df_stat.groupby("unique_id")["ds"].min()
    grid = pd.date_range(TRAIN_START, TRAIN_END, freq="MS")
    tmp = pd.MultiIndex.from_product([first_seen.index, grid],
                                     names=["unique_id", "ds"]).to_frame(index=False)
    tmp = tmp[tmp.ds >= tmp.unique_id.map(first_seen)]
    full_idx = pd.MultiIndex.from_frame(tmp)
    del tmp
    df_stat = (df_stat.set_index(["unique_id", "ds"])
               .reindex(full_idx, fill_value=0.0)
               .reset_index())
    logger.info(f"  Reindexed from each series' first observed month: "
                f"{len(df_stat):,} rows ({df_stat.unique_id.nunique():,} series)")
    # every series must now end December 2016, so forecasts land on Jan-Jun 2017
    assert (df_stat.groupby("unique_id")["ds"].max() ==
            pd.Timestamp("2016-12-01")).all(), "series do not all end Dec 2016"
    # inter-demand intervals should now be visible to Croston/SBA
    _gap = (df_stat[df_stat.y > 0].groupby("unique_id")["ds"]
            .apply(lambda s: s.diff().dt.days.div(30.4).round().mean()))
    logger.info(f"  Mean inter-demand interval: {_gap.mean():.2f} months  "
                f"({(_gap > ADI_THRESH).mean()*100:.1f}% above threshold)")
    del _gap
    df_sbc["unique_id"] = (df_sbc["store_nbr"].astype(str) + "_" + df_sbc["item_nbr"].astype(str))
    def fit_stat(ids, model, label):
        """Fit one statsforecast model. Series with < MIN_OBS months of demand
        fall back to their mean."""
        if not ids:
            return pd.DataFrame(columns=["unique_id", "ds", "pred"])
        df_sub = df_stat[df_stat.unique_id.isin(ids)]
        # series have variable length after reindexing, so count months with
        # actual demand rather than rows -otherwise MIN_OBS would never be used
        n_obs = (df_sub[df_sub.y > 0].groupby("unique_id")["y"].count()
                 .reindex(ids, fill_value=0))
        valid_ids = n_obs[n_obs >= MIN_OBS].index.tolist()
        tiny_ids  = n_obs[n_obs <  MIN_OBS].index.tolist()
        logger.info(f"  {label}: {len(valid_ids):,} fitted  {len(tiny_ids):,} mean fallback")
        results = []
        if valid_ids:
            sf = StatsForecast(models=[model], freq="MS", n_jobs=-1)
            fc = sf.forecast(df=df_sub[df_sub.unique_id.isin(valid_ids)],
                             h=MAX_H).reset_index(drop=True)
            col = [c for c in fc.columns if c not in ["unique_id", "ds"]][0]
            fc  = fc.rename(columns={col: "pred"})[["unique_id", "ds", "pred"]]
            fc["pred"] = fc["pred"].clip(lower=0)
            results.append(fc)
        if tiny_ids:
            means = df_sub[df_sub.unique_id.isin(tiny_ids)].groupby("unique_id")["y"].mean()
            rows  = [{"unique_id": uid, "ds": m, "pred": max(0.0, float(means.get(uid, 0.0)))}
                     for uid in tiny_ids for m in future_months]
            results.append(pd.DataFrame(rows))
        return pd.concat(results, ignore_index=True)
    smooth_ids     = df_sbc.loc[df_sbc.category == "smooth", "unique_id"].tolist()
    non_smooth_ids = df_sbc.loc[df_sbc.category == "non-smooth", "unique_id"].tolist()

    sf_ets = StatsForecast(models=[AutoETS(season_length=12)], freq="MS", n_jobs=-1)
    sf_ets.fit(df=df_stat[df_stat.unique_id.isin(smooth_ids)])
    methods = {uid: sf_ets.fitted_[i, 0].model_["method"] for i, uid in enumerate(sf_ets.uids)}
    selected_model = pd.Series(methods).apply(lambda m: "AutoETS_seas" if is_seasonal(m) else "AutoETS_ns")
    logger.info(f"  AutoETS(s=12): {(selected_model=='AutoETS_seas').sum():,} seasonal, "
                f"{(selected_model=='AutoETS_ns').sum():,} non-seasonal")

    fc_ets = fit_stat(smooth_ids,     AutoETS(season_length=12), "AutoETS(s=12)")
    fc_sba = fit_stat(non_smooth_ids, CrostonSBA(), "CrostonSBA")
    df_stat_fc = pd.concat([fc_ets, fc_sba], ignore_index=True)
    # Croston has no demand size to estimate for an all-zero series and can
    # return NaN; nothing downstream would catch it
    n_nan = df_stat_fc["pred"].isna().sum()
    if n_nan:
        logger.warning(f"  {n_nan:,} NaN predictions -> 0")
        df_stat_fc["pred"] = df_stat_fc["pred"].fillna(0)
    split = df_stat_fc["unique_id"].str.split("_", n=1, expand=True)
    df_stat_fc["store_nbr"] = split[0].astype(int)
    df_stat_fc["item_nbr"]  = split[1].astype(int)
    df_stat_fc["month"]     = pd.to_datetime(df_stat_fc["ds"])
    df_stat_fc = df_stat_fc[["store_nbr", "item_nbr", "month", "pred"]]

    # patch series whose forecast is shorter than MAX_H using their own mean.
    coverage   = df_stat_fc.groupby(["store_nbr", "item_nbr"])["month"].count()
    incomplete = coverage[coverage < MAX_H]
    if len(incomplete) > 0:
        means_patch = df_stat_fc.groupby(["store_nbr", "item_nbr"])["pred"].mean()
        patch_rows  = []
        for (s, i), _ in incomplete.items():
            existing = set(df_stat_fc[(df_stat_fc.store_nbr == s) &
                                      (df_stat_fc.item_nbr  == i)]["month"])
            mp = float(means_patch.get((s, i), 0.0))
            for m in future_months:
                if m not in existing:
                    patch_rows.append({"store_nbr": s, "item_nbr": i,
                                       "month": m, "pred": mp})
        if patch_rows:
            df_stat_fc = pd.concat([df_stat_fc, pd.DataFrame(patch_rows)], ignore_index=True)
        logger.info(f"  Patched {len(patch_rows):,} incomplete rows")
    df_stat_fc.to_csv(stat_pred_path, index=False)

    # AutoETS models chosen for each series
    pd.DataFrame({"unique_id": methods.keys(), "method": methods.values(),
                  "selected_model": selected_model.values}).to_csv(
        os.path.join(RESULTS_DIR, "autoets_model_selection.csv"), index=False)

    del df_stat, fc_ets, fc_sba
    gc.collect()
logger.info(f"  Statistical models done in {(time.time()-t0)/60:.1f} min")

11:04:51  Fitting statistical models...
11:04:54    Reindexed from each series' first observed month: 5,191,889 rows (162,503 series)
11:05:16    Mean inter-demand interval: 1.23 months  (18.4% above threshold)
11:06:06    AutoETS(s=12): 7,955 seasonal, 55,450 non-seasonal
11:06:06    AutoETS(s=12): 63,405 fitted  0 mean fallback
11:06:54    CrostonSBA: 94,465 fitted  4,633 mean fallback
11:07:00    Statistical models done in 2.2 min


**Machine Learning Benchmark: LGBM recursive multi-step**

In [15]:
##### LGBM training and recursive forecast ##################

lgbm_pred_path = os.path.join(RESULTS_DIR, "lgbm_predictions.csv")

if os.path.exists(lgbm_pred_path):
    logger.info("Cached lgbm_predictions.csv found — loading")
    df_preds = pd.read_csv(lgbm_pred_path, parse_dates=["month"])
else:
    ##### train the single h=1 model
    logger.info("Training LightGBM recursive h=1 model...")
    t0 = time.time()

    df_lgbm_train = df_train.copy()
    df_lgbm_train["y_target"] = df_lgbm_train.groupby(
        ["store_nbr", "item_nbr"])["y"].shift(-1)
    df_lgbm_train = df_lgbm_train.dropna(subset=["y_target"] + FEATURES)

    valid_months = sorted(df_lgbm_train.month.unique())
    val_cut      = valid_months[-VAL_MONTHS]
    df_tr = df_lgbm_train[df_lgbm_train.month <  val_cut]
    df_va = df_lgbm_train[df_lgbm_train.month >= val_cut]
    logger.info(f"  Train: {len(df_tr):,} rows  Val: {len(df_va):,} rows  "
                f"Val from {str(val_cut)[:7]}")

    train_set = lgb.Dataset(df_tr[FEATURES].values.astype(np.float32),
                            label=df_tr["y_target"].values)
    val_set   = lgb.Dataset(df_va[FEATURES].values.astype(np.float32),
                            label=df_va["y_target"].values, reference=train_set)

    model = lgb.train(
        LGBM_PARAMS, train_set,
        num_boost_round=NUM_ROUNDS,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False),
                   lgb.log_evaluation(100)],
    )
    logger.info(f"  Best iteration: {model.best_iteration}  "
                f"Time: {(time.time()-t0)/60:.1f} min")
    model.save_model(os.path.join(RESULTS_DIR, "lgbm_model.txt"))

    del df_lgbm_train, df_tr, df_va, train_set, val_set
    gc.collect()

    #### recursive forecast through the 8 test months ####
    logger.info("Running recursive forecast...")
    t0 = time.time()

    test_months  = sorted(df_test.month.unique())
    series_index = (df_test[["store_nbr", "item_nbr"]].drop_duplicates()
                    .set_index(["store_nbr", "item_nbr"]).index)

    # seed the lag state from December 2016 actuals; series with no December 2016
    # row fall back to their last available training month
    dec2016    = pd.Timestamp("2016-12-01")
    dec_state  = df[df.month == dec2016].set_index(
        ["store_nbr", "item_nbr"])[LAG_ROLL + ["y"]]
    last_state = (df[df.month <= TRAIN_END].sort_values("month")
                  .groupby(["store_nbr", "item_nbr"])[LAG_ROLL + ["y"]].last())

    state   = dec_state.reindex(series_index)
    missing = state["lag_1"].isna()
    if missing.sum() > 0:
        state.loc[missing] = last_state.reindex(series_index[missing]).values
        still_nan = state["lag_1"].isna().sum()
        if still_nan:
            logger.warning(f"  {still_nan:,} series zero-filled (no training history)")
            state = state.fillna(0)
    logger.info(f"  State: {len(state):,} series  "
                f"{missing.sum():,} fallback to last training month")

    all_preds = []

    for step, month in enumerate(test_months, 1):
        month_meta = df_test[df_test.month == month].set_index(["store_nbr", "item_nbr"])
        common     = state.index.intersection(month_meta.index)

        rows = []
        for s, i in common:
            st  = state.loc[(s, i)]
            mt  = month_meta.loc[(s, i)]
            row = {
                "store_nbr"        : s,
                "item_nbr"         : i,
                "month_of_year"    : month.month,
                "year"             : month.year,
                "onpromotion"      : mt["onpromotion"],
                "holidays_count"   : mt["holidays_count"],
                "oil_lag1"         : mt["oil_lag1"],
                "transactions_lag1": mt["transactions_lag1"],
                "city_enc"         : mt["city_enc"],
                "state_enc"        : mt["state_enc"],
                "type_enc"         : mt["type_enc"],
                "cluster"          : mt["cluster"],
                "family_enc"       : mt["family_enc"],
                "class_enc"        : mt["class_enc"],
                "perishable"       : mt["perishable"],
            }
            for lag in LAGS:
                row[f"lag_{lag}"] = st[f"lag_{lag}"]
            for win in ROLL_WINS:
                row[f"roll_mean_{win}"] = st[f"roll_mean_{win}"]
                row[f"roll_std_{win}"]  = st[f"roll_std_{win}"]
            rows.append(row)

        X          = pd.DataFrame(rows)[FEATURES].values.astype(np.float32)
        preds      = model.predict(X)
        preds_orig = np.expm1(np.clip(preds, 0, None))

        for idx, (s, i) in enumerate(common):
            al = float(month_meta.loc[(s, i), "y"])
            all_preds.append({
                "store_nbr"  : s, "item_nbr": i, "month": month,
                "pred_orig"  : float(preds_orig[idx]),
                "actual_orig": float(np.expm1(al)),
            })

            # exact lag shift: lag_1 <- prediction, lag_k <- previous lag_{k-1}
            new_y = float(preds[idx])
            old   = state.loc[(s, i)].copy()
            state.at[(s, i), "lag_1"] = new_y
            for k in range(2, max(LAGS) + 1):
                if k in LAGS:
                    state.at[(s, i), f"lag_{k}"] = float(old[f"lag_{k-1}"])
            # rolling statistics recomputed exactly from the updated lag vector
            for win in ROLL_WINS:
                lv = [state.at[(s, i), f"lag_{k}"] for k in range(1, win + 1)]
                state.at[(s, i), f"roll_mean_{win}"] = float(np.mean(lv))
                state.at[(s, i), f"roll_std_{win}"]  = float(
                    np.std(lv) if len(lv) > 1 else 0.0)

        logger.info(f"  Step {step}/{len(test_months)} ({str(month)[:7]}): "
                    f"{len(common):,} series  [{(time.time()-t0)/60:.1f} min]")

    df_preds = pd.DataFrame(all_preds)
    df_preds.to_csv(lgbm_pred_path, index=False)
    logger.info(f"  Recursive forecast done in {(time.time()-t0)/60:.1f} min")

    del all_preds, state
    gc.collect()


11:10:46  Cached lgbm_predictions.csv found — loading


**Evaluation**

In [16]:
###### metrics ######################################################
# Both metrics are computed at item x store x month level in log1p space,
# pooling every row in the evaluation window into a single mean.
#
#   RMSLE = sqrt(mean((log1p(actual) - log1p(pred))^2))
#
#   MAPE  = mean(|log1p(a) - log1p(p)| / log1p(a)) x 100, non-zero log actuals only
#           the paper's
#
# Evaluation is cumulative: h=3 pools January through March, not March alone

def rmsle_mape(actual, pred):
    a  = np.asarray(actual, dtype=float)
    p  = np.clip(np.asarray(pred, dtype=float), 0, None)
    al = np.log1p(a)
    pl = np.log1p(p)
    rmsle = float(np.sqrt(np.mean((al - pl) ** 2)))
    nz    = al > 0
    mape  = float(np.mean(np.abs(al[nz] - pl[nz]) / al[nz]) * 100) if nz.sum() else float("nan")
    return round(rmsle, 4), round(mape, 2)


df_stat_fc["month"] = pd.to_datetime(df_stat_fc["month"])
test_months_sorted  = sorted(df_preds.month.unique())
results = []

for h_label, h in HORIZONS.items():
    eval_months = test_months_sorted[:h]

    # LGBM
    df_h = df_preds[df_preds.month.isin(eval_months)]
    r, m = rmsle_mape(df_h.actual_orig, df_h.pred_orig)
    results.append({"model": "LightGBM (recursive)", "horizon": h_label,
                    "RMSLE": r, "MAPE(%)": m, "n_rows": len(df_h)})

    # Statistical — predictions are already in original units
    df_se = df_test[df_test.month.isin(eval_months)][
        ["store_nbr", "item_nbr", "month", "y"]].copy()
    df_se = df_se.merge(df_stat_fc, on=["store_nbr", "item_nbr", "month"], how="left")
    n_missing = df_se["pred"].isna().sum()
    if n_missing:
        logger.warning(f"  {h_label}: {n_missing:,} statistical predictions missing -> 0")
    df_se["pred"] = df_se["pred"].fillna(0).clip(lower=0)
    r, m = rmsle_mape(np.expm1(df_se["y"]), df_se["pred"])
    results.append({"model": "Statistical (smooth/non-smooth)", "horizon": h_label,
                    "RMSLE": r, "MAPE(%)": m, "n_rows": len(df_se)})

df_results = pd.DataFrame(results)
df_results.to_csv(os.path.join(RESULTS_DIR, "metrics.csv"), index=False)

**Results**

In [18]:
#### results ######################

W = 92
print("\n" + "=" * W)
print("LightGBM and Statistical Benchmarks — case study 3 - Efat et al. (2024)")
print(f"Corporación Favorita | Monthly | Train 2013-2016 | Test Jan-Aug 2017")
print(f"Series: {n_series:,} total  |  {n_train_series:,} with 2013-2016 training history")
print(f"Items: {df_class_src.item_nbr.nunique():,}  Stores: {df_class_src.store_nbr.nunique()}")
print("=" * W)

print(f"\nSeries classification (monthly ADI, denominator = {n_train_months} "
      f"training months, threshold = {ADI_THRESH}):")
for cat, n in df_sbc["category"].value_counts().items():
    mdl = {"smooth": "AutoETS(s=12)", "non-smooth": "CrostonSBA"}[cat]
    print(f"  {cat:<12}: {n:>7,}  ({n/len(df_sbc)*100:5.1f}%)  -> {mdl}")

print(f"\n{'Model':<34} {'Horizon':<10} {'RMSLE':>8} {'MAPE(%)':>9} {'N Rows':>11}")
print("-" * W)
for _, r in df_results.iterrows():
    print(f"{r['model']:<34} {r['horizon']:<10} {r['RMSLE']:>8.4f} "
          f"{r['MAPE(%)']:>9.2f} {r['n_rows']:>11,}")

print(f"\nPaper (Efat et al. 2024, Tables 4 and 6):")
print(f"{'Model':<34} {'Horizon':<10} {'RMSLE':>8} {'MAPE(%)':>9}")
print("-" * W)
for h, r, m in [("1-month", 0.3166, 2.57), ("3-month", 0.9476, 6.03),
                ("6-month", 1.0237, 9.84)]:
    print(f"{'ATES + GRU-CNN-LSTM (proposed)':<34} {h:<10} {r:>8.4f} {m:>9.2f}")
for h, r, m in [("1-month", 0.6185, 4.62), ("3-month", 1.7642, 7.24),
                ("6-month", 3.9617, 13.41)]:
    print(f"{'ATES (standalone statistical)':<34} {h:<10} {r:>8.4f} {m:>9.2f}")

print(f"\nNotes:")
print(f"  RMSLE : sqrt(mean((log1p(actual)-log1p(pred))^2)) at item x store x month level")
print(f"  MAPE  : log1p scale, non-zero log actuals only. The paper's Eq.29 uses")
print(f"          original units, which returns 59-71% at this granularity because")
print(f"          sparse series dominate the denominator.")
print(f"  Eval  : cumulative — h=1 is January, h=3 is Jan-Mar, h=6 is Jan-Jun 2017")
print(f"  h=12  : not feasible — Kaggle train.csv ends 15 August 2017 (8 test months)")
print(f"  LGBM  : recursive multi-step from a single h=1 model. Global model across")
print(f"          all series, versus the paper's per-store design (54 models).")
print(f"          No perishable weights — the paper uses MSE loss without item weights.")
print(f"  Stat  : univariate — no promotion, holiday, oil or transaction features.")
print(f"          Classification on monthly data; models fitted on monthly data in")
print(f"          original units.")
print(f"  Series: 162,503 from train.csv. The paper states 4,100 x 54 but applies")
print(f"          undisclosed data cleaning, so its effective universe is unknown.")
print("=" * W)
print(f"\nOutputs: {RESULTS_DIR}")


LightGBM and Statistical Benchmarks — case study 3 - Efat et al. (2024)
Corporación Favorita | Monthly | Train 2013-2016 | Test Jan-Aug 2017
Series: 174,685 total  |  162,503 with 2013-2016 training history
Items: 3,890  Stores: 53

Series classification (monthly ADI, denominator = 48 training months, threshold = 1.32):
  non-smooth  :  99,098  ( 61.0%)  -> CrostonSBA
  smooth      :  63,405  ( 39.0%)  -> AutoETS(s=12)

Model                              Horizon       RMSLE   MAPE(%)      N Rows
--------------------------------------------------------------------------------------------
LightGBM (recursive)               1-month      0.6267     12.69     136,852
Statistical (smooth/non-smooth)    1-month      0.5983     12.72     136,852
LightGBM (recursive)               3-month      0.6804     13.28     412,568
Statistical (smooth/non-smooth)    3-month      0.6316     13.44     412,568
LightGBM (recursive)               6-month      0.7972     14.92     832,227
Statistical (smooth/

**Monthly bias and variance and recursive error accumulation**

In [1]:
# S = df_test[["store_nbr","item_nbr","month","y"]].merge(
#         df_stat_fc, on=["store_nbr","item_nbr","month"], how="left")
# S["pred"] = S["pred"].fillna(0).clip(lower=0)
# S["actual"] = np.expm1(S.y)

# L = df_preds[["store_nbr","item_nbr","month","actual_orig","pred_orig"]]

# print(f"{'month':<9} {'LGBM':>8} {'Stat':>8} {'gap':>8} {'LGBM bias':>10} {'Stat bias':>10}")
# for mth in sorted(df_preds.month.unique())[:6]:
#     l = L[L.month == mth]; s = S[S.month == mth]
#     rl = np.sqrt(((np.log1p(l.actual_orig)- np.log1p(l.pred_orig))**2).mean())
#     rs = np.sqrt(((np.log1p(s.actual)- np.log1p(s.pred))**2).mean())
#     bl = (np.log1p(l.pred_orig) - np.log1p(l.actual_orig)).mean()
#     bs = (np.log1p(s.pred) - np.log1p(s.actual)).mean()
#     print(f"{str(mth)[:7]:<9} {rl:>8.4f} {rs:>8.4f} {rl-rs:>8.4f} {bl:>+10.4f} {bs:>+10.4f}")

**Statistical models outperform LGBM in this case study owing to two reasons:**
 * At h=1 the gap is small (0.0285) and reflects shrinkage: as a single global model minimising squared error across 162,503 series, LGBM regresses predictions toward the global mean, under-predicting high volume series and over-predicting low-volume ones. The per series statistical models have no global mean to shrink toward.

 * From h=2 onward a second effect compounds on top: LGBM's recursive forecasting compounds errors at each step of the horizon: each month's prediction is fed back as the lag input for the next. The statistical models make a single-shot prediction for the entire horizon with no feedback path. The evidence is the drift in bias : LGBM's mean log bias deepens from −0.17 in January to −0.32 in June, while the statistical bias shows no trend (−0.05 to +0.03). The RMSLE gap correspondingly widens from 0.0285 at h=1 to 0.176 at h=6.


In [25]:
# reads: lgbm_predictions.csv  (via df_preds)
#        statistical_predictions_adi.csv  (via df_stat_fc)
# both already in session; if starting fresh:
df_preds   = pd.read_csv(f"{RESULTS_DIR}/lgbm_predictions.csv", parse_dates=["month"])
df_stat_fc = pd.read_csv(f"{RESULTS_DIR}/statistical_predictions_adi1.csv", parse_dates=["month"])

S = df_test[["store_nbr","item_nbr","month","y"]].merge(
        df_stat_fc, on=["store_nbr","item_nbr","month"], how="left")
S["pred"] = S["pred"].fillna(0).clip(lower=0)
S["actual"] = np.expm1(S.y)

L = df_preds[["store_nbr","item_nbr","month","actual_orig","pred_orig"]]

print(f"{'month':<9} {'LGBM':>8} {'Stat':>8} {'gap':>8} {'LGBM bias':>10} {'Stat bias':>10} "
      f"{'LGBM var':>9} {'Stat var':>9}")
for mth in sorted(df_preds.month.unique())[:6]:
    l = L[L.month == mth]; s = S[S.month == mth]
    el = np.log1p(l.actual_orig) - np.log1p(l.pred_orig)
    es = np.log1p(s.actual)      - np.log1p(s.pred)
    rl, rs = np.sqrt((el**2).mean()), np.sqrt((es**2).mean())
    print(f"{str(mth)[:7]:<9} {rl:>8.4f} {rs:>8.4f} {rl-rs:>8.4f} "
          f"{-el.mean():>+10.4f} {-es.mean():>+10.4f} {el.var():>9.4f} {es.var():>9.4f}")

month         LGBM     Stat      gap  LGBM bias  Stat bias  LGBM var  Stat var
2017-01     0.6267   0.5983   0.0285    -0.1740    -0.0507    0.3625    0.3554
2017-02     0.6487   0.6185   0.0301    -0.0738    +0.0351    0.4153    0.3814
2017-03     0.7577   0.6749   0.0828    -0.1930    -0.0572    0.5369    0.4523
2017-04     0.8294   0.7187   0.1107    -0.1803    -0.0369    0.6554    0.5151
2017-05     0.9137   0.7339   0.1798    -0.3138    -0.0864    0.7364    0.5312
2017-06     0.9447   0.7686   0.1762    -0.3170    -0.0487    0.7920    0.5883
